# Run Strategy — PoliMillionaire
Ogni cella esegue **una sola strategia** e salva i risultati in `runs_log.json`.  
Esegui sempre la **cella 1 (Setup)** per prima, poi solo le celle che ti interessano.

In [1]:
# ── CELLA 1 — SETUP (esegui sempre questa per prima) ─────────────────────
import os, sys, re, io, json, signal, random, time
import math as _math
import ast as _ast
import operator as _operator
from datetime import datetime
from contextlib import redirect_stdout

import numpy as np
import ollama
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

sys.path.insert(0, os.path.abspath('./'))
from millionaire_client import MillionaireClient, AuthenticationError
import dotenv
dotenv.load_dotenv()

client = MillionaireClient('http://131.175.15.22:51111/')
user   = client.login(os.getenv('USER'), os.getenv('PASSWORD'))
print(f'Login OK. Welcome, {user.username}!')

competitions    = client.competitions.list_all()
COMPETITION_IDS = [c.id for c in competitions]
COMP_NAME       = {c.id: c.name for c in competitions}
LABELS          = ['A', 'B', 'C', 'D']
N_GAMES         = 5
QUESTIONS_PER_GAME = 15
LOG_FILE        = 'runs_log.json'

LLAMA    = 'llama3.1:8b'
MISTRAL  = 'mistral:7b'
LLAMA_S  = 'llama3.2'
DEEPSEEK = 'deepseek-r1:7b-qwen-distill-q4_K_M'
VERBOSE  = True

_MATH_GLOBALS = {
    '__builtins__': __builtins__,
    'math': _math, 'sqrt': _math.sqrt, 'log': _math.log, 'log2': _math.log2,
    'log10': _math.log10, 'exp': _math.exp, 'sin': _math.sin, 'cos': _math.cos,
    'tan': _math.tan, 'pi': _math.pi, 'e': _math.e, 'factorial': _math.factorial,
    'comb': _math.comb, 'perm': _math.perm, 'ceil': _math.ceil, 'floor': _math.floor,
    'abs': abs, 'round': round, 'sum': sum, 'range': range,
}

def _parse_letter(raw, question):
    upper = raw.strip().upper()
    m = re.search(r'ANSWER\s*[:=]\s*([A-D])', upper)
    if m: chosen = m.group(1)
    else:
        m = re.search(r'\b([A-D])\b\s*$', upper)
        chosen = m.group(1) if m else None
    if chosen is None:
        for c in reversed(upper):
            if c in LABELS: chosen = c; break
    if chosen is None:
        print(f"  ⚠️  Unparseable: '{raw[:60]}' — random fallback.")
        return random.choice(question.options).id
    return question.options[LABELS.index(chosen)].id

def _llm(prompt, max_tokens=200, model=None, temperature=0.0):
    resp = ollama.chat(
        model=model or LLAMA,
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': max_tokens, 'temperature': temperature},
    )['message']['content']
    if VERBOSE:
        print(f'\n  ┌── [{model or LLAMA}]')
        for line in resp.splitlines(): print(f'  │ {line}')
        print('  └──')
    return resp

def _run_code(code, timeout=10):
    def _h(s, f): raise TimeoutError()
    signal.signal(signal.SIGALRM, _h)
    signal.alarm(timeout)
    try:
        buf = io.StringIO()
        ns  = dict(_MATH_GLOBALS)
        with redirect_stdout(buf): exec(code, ns)
        return buf.getvalue().strip() or None
    except TimeoutError: print('  ⚠️  Code timed out'); return None
    except Exception as e: print(f'  ⚠️  Exec error: {e}'); return None
    finally: signal.alarm(0)

def play_game(competition_id, strategy_fn, game_index, after_answer=None, before_game=None):
    if before_game: before_game()
    game, correct, times = client.game.start(competition_id), 0, []
    while game.in_progress:
        q = game.current_question
        if not q: break
        t0 = time.time()
        chosen_id = strategy_fn(q)
        chosen_text = next(o.text for o in q.options if o.id == chosen_id)
        result = game.answer(chosen_id)
        times.append(time.time() - t0)
        if result.correct: correct += 1
        if after_answer: after_answer(q, chosen_text, result)
        if result.game_over: break
    acc = correct / QUESTIONS_PER_GAME
    icon = '✅' if game.earned_amount > 0 else '❌'
    print(f'    {icon} Game {game_index} | Level {game.current_level} | Earned: {game.earned_amount:,.0f} | Acc: {acc:.0%}')
    return {
        'game': game_index, 'timestamp': datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        'correct': correct, 'accuracy': round(acc, 4),
        'earned': game.earned_amount, 'reached_level': game.current_level,
        'response_time_avg': round(sum(times)/len(times), 3) if times else 0,
    }

def run_strategy(name, fn, comp_ids=None, after_answer=None, before_game=None):
    target_ids = comp_ids if comp_ids is not None else COMPETITION_IDS
    log = json.loads(open(LOG_FILE).read()) if os.path.exists(LOG_FILE) else {}
    print(f'\n=== {name} | {len(target_ids)} comp × {N_GAMES} games ===')
    for comp_id in target_ids:
        comp_name = COMP_NAME[comp_id]
        print(f'  --- {comp_name} ---')
        games = [play_game(comp_id, fn, i, after_answer=after_answer, before_game=before_game)
                 for i in range(1, N_GAMES + 1)]
        log.setdefault(name, {}).setdefault(comp_name, {'runs': []})
        run_id = len(log[name][comp_name]['runs']) + 1
        log[name][comp_name]['runs'].append({
            'run_id': run_id,
            'games': games,
            'avg_accuracy': round(sum(g['accuracy'] for g in games) / len(games), 4),
            'avg_earned':   round(sum(g['earned']   for g in games) / len(games), 2),
        })
    with open(LOG_FILE, 'w') as f: json.dump(log, f, ensure_ascii=False, indent=2)
    print(f'Salvato in {LOG_FILE}')

print('Setup completo. Esegui la cella della strategia che vuoi testare.')

/Users/aliceberta/Development/NLP/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Login OK. Welcome, Alisss!
Setup completo. Esegui la cella della strategia che vuoi testare.


In [ ]:
# ── CELLA 2 — ZERO-SHOT ──────────────────────────────────────────────────
def zero_shot(question):
    opts = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    return _parse_letter(_llm(
        f'Answer this multiple choice question. Reply with ONLY the letter (A, B, C or D).\n\n'
        f'Question: {question.text}\n\n{opts}\n\nAnswer:',
        max_tokens=10
    ), question)

run_strategy('zero-shot', zero_shot)

In [ ]:
# ── CELLA 3 — COT-FIXED ──────────────────────────────────────────────────
def cot_fixed(question):
    opts = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    return _parse_letter(_llm(
        f'Answer the multiple choice question below.\n'
        f'Reason briefly (2-3 sentences), then on the last line write exactly: Answer: X\n'
        f'(X must be A, B, C, or D)\n\nQuestion: {question.text}\n\n{opts}\n',
        max_tokens=300
    ), question)

run_strategy('cot-fixed', cot_fixed)

In [ ]:
# ── CELLA 4 — SEMANTIC FEW-SHOT con MMLU ─────────────────────────────────
MMLU_SUBJECTS = [
    'high_school_physics', 'high_school_chemistry', 'high_school_biology',
    'high_school_mathematics', 'high_school_statistics', 'college_mathematics',
    'high_school_world_history', 'high_school_european_history', 'prehistory',
    'high_school_geography', 'sociology', 'high_school_macroeconomics',
    'astronomy', 'anatomy', 'nutrition', 'virology',
    'philosophy', 'logical_fallacies', 'high_school_computer_science',
    'high_school_psychology', 'global_facts', 'miscellaneous',
]

embedder = SentenceTransformer('all-MiniLM-L6-v2')
_pairs, _embs = [], None

for subj in MMLU_SUBJECTS:
    try:
        ds = load_dataset('cais/mmlu', subj, split='test')
        for row in ds.select(range(min(25, len(ds)))):
            idx = row['answer']
            _pairs.append((row['question'], f"{LABELS[idx]}) {row['choices'][idx]}"))
    except: pass

if _pairs:
    _embs = embedder.encode([p for p, _ in _pairs], convert_to_numpy=True)
print(f'Bank: {len(_pairs)} esempi MMLU')
_mmlu_checkpoint = (_pairs.copy(), _embs.copy())

def _retrieve(query, top_k=3):
    if not _pairs: return []
    q_emb = embedder.encode([query], convert_to_numpy=True)
    sims  = (_embs / (np.linalg.norm(_embs, axis=1, keepdims=True) + 1e-8)) @ \
            (q_emb / (np.linalg.norm(q_emb) + 1e-8)).T
    return [_pairs[i] for i in np.argsort(sims[:, 0])[::-1][:top_k]]

def semantic_few_shot(question):
    opts = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    examples = _retrieve(question.text)
    ex_block = '\n'.join(f'Question: {q}\nAnswer: {a}\n' for q, a in examples)
    prompt = (
        f'Here are similar questions with correct answers:\n\n{ex_block}\n'
        f'Now answer this. Reply with ONLY the letter (A, B, C or D).\n\n'
        f'Question: {question.text}\n{opts}\nAnswer:'
    ) if examples else (
        f'Answer this. Reply with ONLY the letter (A, B, C or D).\n\n'
        f'Question: {question.text}\n\n{opts}\n\nAnswer:'
    )
    return _parse_letter(_llm(prompt, max_tokens=10), question)

def _bank_reset():
    global _pairs, _embs
    _pairs = list(_mmlu_checkpoint[0])
    _embs  = _mmlu_checkpoint[1].copy()

def _bank_update(question, chosen_text, result):
    global _pairs, _embs
    if result.correct:
        _pairs.append((question.text, chosen_text))
        new_emb = embedder.encode([question.text], convert_to_numpy=True)
        _embs = np.vstack([_embs, new_emb])

run_strategy('semantic-few-shot', semantic_few_shot,
             after_answer=_bank_update, before_game=_bank_reset)

## Strategia 5 — Hybrid-Math

Abbiamo sviluppato cinque versioni successive della strategia hybrid-math, ognuna nata da problemi osservati nell'output reale del gioco.

**v1 — Classificatore LLM:** il classificatore decideva se rispondere con codice Python o con ragionamento concettuale (CALCULATE vs REASON). Il problema era un bias forte verso REASON: anche domande di calcolo puro venivano classificate come concettuali, quindi il path del codice non veniva quasi mai attivato e il modello cercava di ragionare su problemi che avrebbero richiesto calcolo, sbagliando sistematicamente.

**v2 — Codice sempre per primo:** rimosso il classificatore, si prova sempre il path del codice prima usando il reasoning come fallback. Migliorava i problemi numerici, ma per le domande puramente concettuali il modello generava codice irrilevante, e il reasoning fallback riceveva a volte un hint fuorviante.

**v3 — Dettagli tecnici:** aumentati i `max_tokens` del reasoning da 300 a 600; aggiunta l'istruzione anti-loop al prompt (`do not repeat yourself, if unsure commit to the most likely answer anyway`); migliorato il parsing delle opzioni con conversione diretta `float()` (risolve il mancato matching di valori semplici come `3` o `0.5`).

**v4 — Validazione dell'hint:** il valore calcolato viene passato come hint al reasoning solo se rientra nell'intorno `[min_opt − range, max_opt + range]` delle opzioni disponibili. Evita che il modello si avviti su valori palesemente sbagliati (es. −11.05 per una domanda sulle percentuali).

**v5 — Header automatico:** prima dell'esecuzione, un header viene preposto al codice generato con tutti gli import più comunemente dimenticati (`reduce`, `combinations`, `permutations`, `numpy`, `scipy.stats`, `Fraction`). Risolve i NameError silenziosi — in particolare la domanda del lancio dei dadi veniva sbagliata ogni volta per un `NameError` su `reduce`.

In [2]:
# ── CELLA 6 — HYBRID-MATH (con classificatore) ───────────────────────────
_CLASSIFY_PROMPT = """\
Does this question require computing a number, or knowing a concept?
Reply with exactly one word: CALCULATE or REASON.

CALCULATE examples:
- "John had 66 pins left after giving away half of a third of a pile. How many did he start with?" → CALCULATE
- "A student scored 58 on a midterm with mean 52 and std 4. Find her final exam score." → CALCULATE
- "How many edges does a complete graph on 10 vertices have?" → CALCULATE
- "What is the probability that..." → CALCULATE

REASON examples:
- "Which of the following is the definition of a prime number?" → REASON
- "Statement 1: Every field is a ring. Statement 2: ..." → REASON
- "Which of the following best describes a Type II error?" → REASON

Question: {question}"""

def _classify(question) -> bool:
    resp = _llm(_CLASSIFY_PROMPT.format(question=question.text), max_tokens=10)
    return 'CALCULATE' in resp.strip().upper()

def hybrid_math(question) -> int:
    opts_str = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))

    if _classify(question):
        raw_code = _llm(_CODE_PROMPT.format(question=question.text, options=opts_str), max_tokens=600)
        code     = _extract_code(raw_code)
        output   = _run_code_subprocess(code)
        if output:
            matched = _match_to_option(output, question)
            if matched:
                return matched
            # Calcolo riuscito ma non matcha esattamente le opzioni — usa come hint
            augmented = (
                f"{_REASON_PROMPT.format(question=question.text, options=opts_str)}\n\n"
                f"Hint: a calculation gave: {output}"
            )
            return _letter_to_id(_to_letter(_llm(augmented, max_tokens=200)), question)

    # Ragionamento concettuale (o fallback se il codice non ha prodotto output)
    resp = _llm(_REASON_PROMPT.format(question=question.text, options=opts_str), max_tokens=500)
    return _letter_to_id(_to_letter(resp), question)

run_strategy('hybrid-math-llama', hybrid_math, comp_ids=[3])


=== hybrid-math-llama | 1 comp × 5 games ===
  --- Maths ---

  ┌── [llama3.1:8b]
  │ CALCULATE
  └──


NameError: name '_CODE_PROMPT' is not defined

In [ ]:
# ── CELLA 7 — HYBRID-MATH-2 (senza classificatore) ───────────────────────
#
# ── MODIFICHE v3 → v4 ─────────────────────────────────────────────────────
# 1. LIBRERIE: scipy e numpy elencati esplicitamente nel code prompt.
# 2. VALIDAZIONE DELL'HINT: hint passato al reasoning solo se il valore
#    calcolato rientra in [min_opt − range, max_opt + range].
# 3. ANTI-LOOP: "Do not repeat yourself. If unsure, commit anyway."
#
# ── MODIFICHE v4 → v5 ─────────────────────────────────────────────────────
# 4. HEADER AUTOMATICO: header preposto al codice con import comuni
#    dimenticati (reduce, combinations, permutations, numpy, scipy, Fraction).
# ──────────────────────────────────────────────────────────────────────────

def hybrid_math(question) -> int:
    opts_str = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))

    # ── DEBUG: stampa domanda e opzioni ───────────────────────────────────
    print(f"\n❓ {question.text}")
    print(opts_str)
    print()
    # ──────────────────────────────────────────────────────────────────────

    # Prova sempre il path di calcolo prima — niente classificatore
    raw_code = _llm(_CODE_PROMPT.format(question=question.text, options=opts_str), max_tokens=600)
    code     = _extract_code(raw_code)
    output   = _run_code_subprocess(code)

    if output:
        matched = _match_to_option(output, question)
        if matched:
            return matched

        # Usa l'hint solo se il valore calcolato è plausibile rispetto alle opzioni
        if _hint_is_plausible(output, question):
            augmented = (
                f"{_REASON_PROMPT.format(question=question.text, options=opts_str)}\n\n"
                f"Hint: a calculation gave: {output}"
            )
            return _letter_to_id(_to_letter(_llm(augmented, max_tokens=300)), question)

    # Fallback: ragionamento concettuale puro
    resp = _llm(_REASON_PROMPT.format(question=question.text, options=opts_str), max_tokens=600)
    return _letter_to_id(_to_letter(resp), question)

run_strategy('hybrid-math-llama', hybrid_math, comp_ids=[3])

## CELLA 8 — DEEPSEEK 

In [ ]:
# ── CELLA 8 — DEEPSEEK ───────────────────────────────────────────────────
def deepseek(question):
    opts = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    if VERBOSE:
        print(f"\n❓ {question.text}")
        print(opts)

    t0 = time.time()
    raw = _llm(
        f'Answer this multiple choice question.\n'
        f'Think briefly, then reply with ONLY the letter (A, B, C or D) on the last line.\n\n'
        f'Question: {question.text}\n\n{opts}\n\nAnswer:',
        max_tokens=1000, model=DEEPSEEK
    )
    print(f'  ⏱️  {time.time() - t0:.1f}s')

    clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    if not clean or '<think>' in clean:
        lines = [l.strip() for l in raw.splitlines() if l.strip()]
        clean = '\n'.join(lines[-3:])

    return _parse_letter(clean, question)

run_strategy('deepseek', deepseek, comp_ids=[3])

## Strategia 8 — Hybrid-Math-DeepSeek

Evoluzione finale per la categoria matematica: **Llama 3.1 8b** genera il codice Python, **DeepSeek-R1 1.5b** si occupa del reasoning.

**Architettura:** Llama genera sempre il codice per primo. Se l'output numerico corrisponde a un'opzione (tolleranza 1%), risposta immediata. Se il valore esiste ma non matcha, viene passato come hint a DeepSeek (purché plausibile rispetto al range delle opzioni). Se il codice non produce output utile, DeepSeek ragiona in autonomia.

**Novità principale — `_deepseek_parse`:** gestisce i casi di thinking troncato cercando prima il pattern `Answer: X` ovunque nel testo grezzo, e come fallback scansiona il testo al contrario per l'ultima lettera valida. Risolve il problema degli output vuoti delle versioni precedenti. `max_tokens=2000` dà al modello spazio sufficiente per completare sia il ragionamento che la risposta.

**Risultati:** Game 1 ha raggiunto il livello 6 con il 33% di accuracy — il risultato migliore sull'intera categoria matematica nei test. La varianza tra le partite rimane elevata (0–33%) a seconda della distribuzione delle domande; algebra astratta avanzata e calcolo multivariabile rimangono problematici per entrambi i modelli. Questa varianza è la motivazione principale per procedere con il fine-tuning su MMLU.

In [ ]:
# ── CELLA 9 — HYBRID-MATH-DEEPSEEK ───────────────────────────────────────
_DS_REASON_PROMPT = """\
Answer this multiple choice question.
Think carefully, then on the LAST line write exactly: Answer: X
(X is A, B, C or D — nothing else after it)

Question: {question}

{options}"""

def _deepseek_parse(raw: str, question) -> int:
    clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    if not clean or '<think>' in clean:
        m = re.search(r'Answer\s*:\s*([A-D])', raw, re.IGNORECASE)
        if m: return _letter_to_id(m.group(1).upper(), question)
        for ch in reversed(raw.upper()):
            if ch in 'ABCD': return _letter_to_id(ch, question)
        return _letter_to_id('A', question)
    return _letter_to_id(_to_letter(clean), question)

def hybrid_math_deepseek(question) -> int:
    opts_str = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    if VERBOSE:
        print(f"\n❓ {question.text}")
        print(opts_str)

    raw_code = _llm(_CODE_PROMPT.format(question=question.text, options=opts_str),
                    max_tokens=600, model=LLAMA)
    code   = _extract_code(raw_code)
    output = _run_code_subprocess(code)

    if output:
        matched = _match_to_option(output, question)
        if matched: return matched
        if _hint_is_plausible(output, question):
            augmented = (
                f"{_DS_REASON_PROMPT.format(question=question.text, options=opts_str)}\n\n"
                f"Hint: a calculation gave: {output}"
            )
            raw = _llm(augmented, max_tokens=2000, model=DEEPSEEK)
            return _deepseek_parse(raw, question)

    raw = _llm(_DS_REASON_PROMPT.format(question=question.text, options=opts_str),
               max_tokens=2000, model=DEEPSEEK)
    return _deepseek_parse(raw, question)

run_strategy('hybrid-math-deepseek', hybrid_math_deepseek, comp_ids=[3])

## Strategia 10 — Fine-tuned Llama 3.1 8B

**Risultati offline:** **67.20%** sulle 47 domande reali di PoliMillionaire (held-out, mai viste in training).

Usa `mlx_lm.generate()` — non passa per ollama. Solo competizione **Maths** (`comp_ids=[3]`).

In [ ]:
# ── CELLA — FINE-TUNED (adapter_v2) ───────────────────────────────────

from mlx_lm import load as _mlx_load, generate as _mlx_generate

print("Carico il modello fine-tuned v2 (adapter_v2)...")
_ft_model, _ft_tokenizer = _mlx_load(
    "mlx-community/Meta-Llama-3.1-8B-Instruct-4bit",
    adapter_path="../fine-tuning/adapter_v2"
)
print("Modello pronto.")

_FT_SYSTEM_V2 = (
    "You are a mathematics expert. "
    "Given a multiple-choice question with four options (A, B, C, D), "
    "reason step by step and always end your answer with: "
    "'Therefore, the correct answer is [LETTER].' "
)

def _ft_v2_build_prompt(question, opts_str):
    user_msg = f"Question: {question.text}\n{opts_str}"
    return (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f'{_FT_SYSTEM_V2}<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'{user_msg}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )

def _ft_v2_extract_letter(text):
    m = re.search(r'correct answer is\s+([A-D])', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    for char in reversed(text.strip().upper()):
        if char in 'ABCD': return char
    return None

def finetuned_v2(question):
    opts_str = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    if VERBOSE:
        print(f"\n❓ {question.text}")
        print(opts_str)

    prompt = _ft_v2_build_prompt(question, opts_str)
    output = _mlx_generate(_ft_model, _ft_tokenizer, prompt=prompt, max_tokens=400, verbose=False)

    if VERBOSE:
        print(f'\n  ┌── [finetuned-v2]')
        for line in output.strip().splitlines(): print(f'  │ {line}')
        print(f'  └──')

    letter = _ft_v2_extract_letter(output)
    if letter is None:
        print("  ⚠️  Nessuna lettera estratta — random fallback.")
        return random.choice(question.options).id
    return question.options[LABELS.index(letter)].id

run_strategy('finetuned-v2', finetuned_v2, comp_ids=[3])

Carico il modello fine-tuned v2 (adapter_v2)...


Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 124583.29it/s]


Modello pronto.

=== finetuned-v2 | 1 comp × 5 games ===
  --- Maths ---

❓ Suppose the graph of $y=f(x)$ includes the points $(1,5),$ $(2,3),$ and $(3,1)$. Based only on this information, there are two points that must be on the graph of $y=f(f(x))$. If we call those points $(a,b)$ and $(c,d),$ what is $ab+cd$?
A) 17
B) 15
C) 9
D) 5


In [ ]:
# ── CELLA 11 — CODE-THEN-FINETUNED ───────────────────────────────────────
# Llama base (ollama) genera il codice → se non basta, fallback al fine-tuned MLX.
# Richiede che la cella 5 (helper) e la cella 10 (fine-tuned) siano già eseguite.

def code_then_finetuned(question) -> int:
    opts_str = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    if VERBOSE:
        print(f"\n❓ {question.text}")
        print(opts_str)

    # STEP 1: Llama base genera codice Python
    raw_code = _llm(
        _CODE_PROMPT.format(question=question.text, options=opts_str),
        max_tokens=600, model=LLAMA
    )
    code   = _extract_code(raw_code)
    output = _run_code_subprocess(code)

    if VERBOSE:
        print(f'  │ Output script: {repr(output)}')

    if output:
        # STEP 2: match diretto → risposta immediata
        matched = _match_to_option(output, question)
        if matched:
            if VERBOSE: print(f'  └── ✅ Risposta da codice')
            return matched

        # STEP 3: hint plausibile → fine-tuned ragiona con il valore calcolato
        if _hint_is_plausible(output, question):
            if VERBOSE: print(f'  │ 💡 Hint plausibile → fine-tuned con hint')
            prompt = _ft_build_prompt(question, opts_str, hint=output)
            out_ft = _mlx_generate(_ft_model, _ft_tokenizer, prompt=prompt, max_tokens=400, verbose=False)
            if VERBOSE:
                print(f'\n  ┌── [finetuned-llama31 + hint]')
                for line in out_ft.strip().splitlines(): print(f'  │ {line}')
                print(f'  └──')
            letter = _ft_extract_letter(out_ft)
            if letter:
                return question.options[LABELS.index(letter)].id

    # STEP 4: codice non conclusivo → fine-tuned puro
    if VERBOSE: print(f'  │ ⚠️  Codice non conclusivo → fine-tuned puro')
    return finetuned(question)

run_strategy('code-then-finetuned', code_then_finetuned, comp_ids=[3])

## 12: ENSEMBLE

In [ ]:
# ── CELLA 12 — ENSEMBLE-3 ────────────────────────────────────────────────
def ensemble(question):
    opts   = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    prompt = (
        f'Answer this multiple choice question. Reply with ONLY the letter (A, B, C or D).\n\n'
        f'Question: {question.text}\n\n{opts}\n\nAnswer:'
    )
    votes = {}
    for m in [LLAMA, MISTRAL, LLAMA_S]:
        opt_id = _parse_letter(_llm(prompt, max_tokens=10, model=m), question)
        votes[opt_id] = votes.get(opt_id, 0) + 1
    max_v   = max(votes.values())
    winners = [k for k, v in votes.items() if v == max_v]
    if len(winners) == 1: return winners[0]
    return _parse_letter(_llm(prompt, max_tokens=10, model=LLAMA), question)

run_strategy('ensemble-3', ensemble)